# Potential-field radial spectrum lab
A synthetic check of annular mean Fourier amplitude and bounded single-layer fitting. Wavenumbers are angular; depths use the grid-spacing unit.

In [ ]:
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / 'implementations' / 'python' / 'potential-field'))
from radial_spectrum import radial_average_amplitude, fit_single_layer_spectrum

In [ ]:
rng = np.random.default_rng(7)
shape = (96, 96)
ky = 2*np.pi*np.fft.fftfreq(shape[0], d=1.0)
kx = 2*np.pi*np.fft.fftfreq(shape[1], d=1.0)
k = np.sqrt(ky[:, None]**2 + kx[None, :]**2)
B_true, z_true, n_true = 2.5, 3.2, 0
spectrum = np.zeros(shape, dtype=complex)
positive = k > 0
amp = B_true * k[positive]**n_true * np.exp(-k[positive]*z_true)
spectrum[positive] = amp * np.exp(1j*rng.uniform(-np.pi, np.pi, amp.size))
grid = np.fft.ifft2(spectrum).real
radial = radial_average_amplitude(grid, 1, 1, n_bins=30, min_count=2)
fit = fit_single_layer_spectrum(radial['wavenumber'], radial['amplitude'], exponents=(0,))
print('true depth:', z_true, 'estimated depth:', fit['z'])
print('true B:', B_true, 'estimated B:', fit['B'])
print('log RMSE:', fit['rmse_log'])

In [ ]:
import matplotlib.pyplot as plt
plt.semilogy(radial['wavenumber'], radial['amplitude'], 'o', label='annular mean amplitude')
plt.semilogy(fit['wavenumber'], fit['fitted_amplitude'], '-', label='single-layer fit')
plt.xlabel('Radial angular wavenumber')
plt.ylabel('Fourier amplitude')
plt.legend()
plt.show()

## Interpretation limits
This is a controlled single-layer synthetic exercise. Real spectra can contain overlapping source ensembles, anisotropy, correlated Fourier coefficients, noise, regional trends and edge effects. Choosing a straight segment and exponent is a modelling decision. The fitter does not solve the nonlinear multi-layer Phillips model or provide uncertainty intervals.